# Fuel Price EDA

Exploratory analysis of `data/silver/prices_silver.parquet`.
One row = one unique price-change event, enriched with station details.

Sections: grade coverage, price distributions, brand, station type, price staleness, regional patterns, diesel-petrol spread.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

df = pd.read_parquet("data/silver/prices_silver.parquet")
n_rows, n_stations = len(df), df["node_id"].nunique()
print(f"Rows: {n_rows:,}  |  Unique stations: {n_stations:,}")
df.head(3)

## 1. Grade coverage

How many stations sell each grade? Establishes what we can actually model.

In [ ]:
grade_counts = (
    df.groupby("fuel_type")["node_id"]
    .nunique()
    .sort_values(ascending=False)
    .rename("stations")
    .reset_index()
)
total = df["node_id"].nunique()
grade_counts["pct_of_all_stations"] = (grade_counts["stations"] / total * 100).round(1)
grade_counts

## 2. Price distributions

Current E10 and B7_STANDARD price distributions across all stations.
Dashed line = mean, dotted = median.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, grade, color in zip(axes, ["E10", "B7_STANDARD"], ["steelblue", "coral"]):
    data = df.loc[df["fuel_type"] == grade, "price_ppl"].dropna()
    ax.hist(data, bins=60, color=color, edgecolor="none", alpha=0.85)
    mean, median = data.mean(), data.median()
    ax.axvline(mean,   color="black", linestyle="--", lw=1.2, label=f"Mean:   {mean:.1f}p")
    ax.axvline(median, color="grey",  linestyle=":",  lw=1.2, label=f"Median: {median:.1f}p")
    ax.set_title(grade)
    ax.set_xlabel("Price (pence per litre)")
    ax.set_ylabel("Stations")
    ax.legend(fontsize=9)

fig.suptitle("Price distributions", fontsize=13)
plt.tight_layout()
plt.show()

for grade in ["E10", "B7_STANDARD"]:
    print()
    print(grade)
    print(df.loc[df["fuel_type"] == grade, "price_ppl"].describe().round(2).to_string())

## 3. Brand analysis

Median E10 price for the 20 largest brands by station count.
The fair-price model needs to control for brand: supermarkets structurally price below independents.

In [ ]:
e10 = df[df["fuel_type"] == "E10"].dropna(subset=["brand_name"])

brand_stats = (
    e10.groupby("brand_name")["price_ppl"]
    .agg(stations="count", mean="mean", median="median", std="std")
    .reset_index()
)
top20 = brand_stats.nlargest(20, "stations")["brand_name"]
plot_data = brand_stats[brand_stats["brand_name"].isin(top20)].sort_values("median")

overall_median = e10["price_ppl"].median()

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(plot_data["brand_name"], plot_data["median"], color="steelblue", alpha=0.8)
ax.axvline(overall_median, color="red", lw=1.2, linestyle="--",
           label=f"Overall median: {overall_median:.1f}p")
ax.set_xlabel("Median E10 price (pence per litre)")
ax.set_title("Median E10 price by brand (top 20 by station count)")
ax.legend()
plt.tight_layout()
plt.show()

print(
    brand_stats.nlargest(20, "stations")
    [["brand_name", "stations", "mean", "median", "std"]]
    .sort_values("median")
    .round(1)
    .to_string(index=False)
)

## 4. Station type: motorway vs supermarket vs independent

Three structurally different groups with different cost bases and customer captivity.
The model must treat them separately.

In [ ]:
e10 = df[df["fuel_type"] == "E10"].copy()

def label_type(row):
    if row["is_motorway"] is True:
        return "Motorway"
    if row["is_supermarket"] is True:
        return "Supermarket"
    return "Independent"

e10["station_type"] = e10.apply(label_type, axis=1)

fig, ax = plt.subplots(figsize=(10, 4))
palette = {"Motorway": "firebrick", "Supermarket": "steelblue", "Independent": "grey"}
for stype, color in palette.items():
    data = e10.loc[e10["station_type"] == stype, "price_ppl"]
    ax.hist(data, bins=50, alpha=0.55,
            label=f"{stype} (n={len(data):,})", color=color, density=True)

ax.set_xlabel("E10 price (pence per litre)")
ax.set_ylabel("Density")
ax.set_title("E10 price distribution by station type")
ax.legend()
plt.tight_layout()
plt.show()

print(e10.groupby("station_type")["price_ppl"].describe().round(2))

## 5. Price staleness

How long ago did each station last change its price?
Stations that rarely change are less useful for modelling price dynamics.

In [ ]:
now = pd.Timestamp.now(tz="UTC")
e10 = df[df["fuel_type"] == "E10"].copy()
e10["days_since_change"] = (now - e10["price_change_effective_timestamp"]).dt.days

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(e10["days_since_change"].clip(upper=180), bins=60,
        color="steelblue", alpha=0.85, edgecolor="none")
ax.set_xlabel("Days since last price change (capped at 180)")
ax.set_ylabel("Stations")
ax.set_title("How long ago did each E10 station last change its price?")
plt.tight_layout()
plt.show()

d = e10["days_since_change"]
print(d.describe().round(1).to_string())
print()
print(f"Unchanged for >30 days : {(d > 30).sum():,}")
print(f"Unchanged for >7 days  : {(d > 7).sum():,}")
print(f"Changed within 24h     : {(d <= 1).sum():,}")

## 6. Regional patterns

Average E10 price by country. A national model may need country-level fixed effects if regional variation is large.

In [ ]:
country_stats = (
    df[df["fuel_type"] == "E10"]
    .dropna(subset=["country"])
    .groupby("country")["price_ppl"]
    .agg(stations="count", mean="mean", median="median", std="std")
    .sort_values("median", ascending=False)
    .reset_index()
)
print("E10 prices by country")
print()
print(country_stats.round(2).to_string(index=False))

## 7. Diesel-petrol spread (B7_STANDARD minus E10)

For stations selling both grades: how much more expensive is diesel?
A consistent spread suggests a structural premium. Negative values are anomalies.

In [ ]:
e10_p = (
    df[df["fuel_type"] == "E10"][["node_id", "price_ppl"]]
    .rename(columns={"price_ppl": "e10"})
)
b7_p = (
    df[df["fuel_type"] == "B7_STANDARD"][["node_id", "price_ppl"]]
    .rename(columns={"price_ppl": "b7"})
)
spread = e10_p.merge(b7_p, on="node_id")
spread["spread_ppl"] = spread["b7"] - spread["e10"]

mean_spread = spread["spread_ppl"].mean()
n_both = len(spread)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(spread["spread_ppl"], bins=60, color="steelblue", alpha=0.85, edgecolor="none")
ax.axvline(mean_spread, color="black", lw=1.2, linestyle="--",
           label=f"Mean: {mean_spread:.1f}p")
ax.set_xlabel("B7_STANDARD minus E10 (pence per litre)")
ax.set_ylabel("Stations")
ax.set_title(f"Diesel-petrol spread across {n_both:,} stations selling both grades")
ax.legend()
plt.tight_layout()
plt.show()

print("Spread (B7_STANDARD - E10, pence per litre)")
print(spread["spread_ppl"].describe().round(2).to_string())
print()
n_negative = (spread["spread_ppl"] < 0).sum()
print(f"Stations where diesel < petrol: {n_negative}")